# NDE: Neural Developmental Encodings

In [1]:
# Standard library
from math import isclose

# Third party libraries
import numpy as np
import torch

# Local libraries
from ariel import console
from ariel.ec.genotypes.nde import NeuralDevelopmentalEncoding

In [2]:
# --- RANDOM GENERATOR SETUP --- #
SEED = 42
RNG = np.random.default_rng(SEED)  # the genes
torch.manual_seed(SEED)  # the starting weights of every network

In [3]:
# Global variables
NUM_OF_MODULES = 30
SAMPLES = 500

## Diversity Assessment

An NDE genotype is three chromosomes of genes; a network turns them
into three probability matrices (module types, connections,
rotations), which a decoder later turns into a body. Each sample here
draws the genes uniformly from a range, as `A3_template.py` does, and
decodes them with a freshly initialised network. The diversity of 500
samples is the mean L1 distance between the flattened matrices of two
samples, over all 124,750 pairs.

In [4]:
def decoded(low: float, high: float, genotype_size: int = 64) -> np.ndarray:
    """Decode ``SAMPLES`` random genotypes, each with a new network.

    Parameters
    ----------
    low, high : float
        The range the genes are drawn from, uniformly.
    genotype_size : int
        The number of genes in each of the three chromosomes.

    Returns
    -------
    np.ndarray
        One row per sample: its three probability matrices, flattened.
    """
    rows = []
    for _ in range(SAMPLES):
        genotype = [
            RNG.uniform(low, high, genotype_size).astype(np.float32)
            for _ in range(3)
        ]
        nde = NeuralDevelopmentalEncoding(
            number_of_modules=NUM_OF_MODULES,
            genotype_size=genotype_size,
        )
        matrices = nde.forward(genotype)
        rows.append(np.concatenate([p.flatten() for p in matrices]))
    return np.array(rows)


def diversity(rows: np.ndarray) -> float:
    """Average the L1 distance between two rows over all pairs of rows.

    Returns
    -------
    float
        The mean distance; 0 when all rows are equal.
    """
    total = sum(
        float(np.abs(rows[i + 1 :] - rows[i]).sum())
        for i in range(len(rows) - 1)
    )
    return total / (len(rows) * (len(rows) - 1) / 2)

### Genes in [0, 1)

In [5]:
unit_rows = decoded(0, 1)
unit = diversity(unit_rows)
console.print(f"[bold green]Mean diversity:[/bold green] {unit:.2f}")

Mean diversity: 100.81

### Genes in [-8192, 8192]

In [6]:
scale = 8192
wide = diversity(decoded(-scale, scale))
console.print(f"[bold green]Mean diversity:[/bold green] {wide:.2f}")

Mean diversity: 178.87

In [7]:
# Checks: every decoded value is a probability (the network ends in a
# sigmoid), and genes drawn from the wider range give more diverse
# matrices
assert ((unit_rows > 0) & (unit_rows < 1)).all()
assert 0 < unit < wide, (unit, wide)

### Find Maximal Diversity

Double the range, with 256 genes per chromosome, until the diversity
stops changing by more than 1.

In [8]:
history: dict[int, float] = {}
old_diversity, new_diversity = 0, -1
scale = 1
while not isclose(old_diversity, new_diversity):
    scale *= 2
    old_diversity = new_diversity
    history[scale] = diversity(decoded(-scale, scale, genotype_size=256))
    msg = f"Scale {scale}: [bold green]mean diversity[/bold green] "
    console.print(msg + f"{history[scale]:.2f}")
    new_diversity = int(history[scale])

Scale 2: mean diversity 105.57

Scale 4: mean diversity 118.46

Scale 8: mean diversity 137.56

Scale 16: mean diversity 155.28

Scale 32: mean diversity 167.16

...

Scale 256: mean diversity 178.86

Scale 512: mean diversity 179.56

Scale 1024: mean diversity 178.51

Scale 2048: mean diversity 179.59

Scale 4096: mean diversity 179.55

In [9]:
# Checks: diversity grows with the range, and the search stopped when
# one doubling left its whole part unchanged
values = list(history.values())
assert values[-1] > values[0], history
assert int(values[-1]) == int(values[-2]), history

## Conclusion

The range the genes are drawn from sets how far apart the decoded
bodies are. Genes in [0, 1) decode to matrices that differ by 100.8 on
average; genes in [-8192, 8192] by 178.9, 1.8 times as much. With 256
genes per chromosome, doubling the range from [-2, 2] raises the
diversity from 105.6 to 178.9 at [-256, 256], and after that it barely
moves (179.6 at [-4096, 4096]). So genes drawn from [0, 1) spread the
decoded matrices only about half as far as the encoding allows, and a
range of a few hundred already reaches nearly the whole spread.